# 期待値・MFE/MAE・決済の分析

**Historical archive / 過去の研究記録**

原本のコードを保持しています。独立実行や現在の検証基準への適合は保証しません。前のセルの変数に依存する箇所があります。実行入口は `../08_trade_quality.ipynb` を参照してください。

保存出力は `../../results/legacy/`、既知の問題は `../../docs/AUDIT.md` に整理しています。


## 元Notebookのセル 12

出典: `FX.ipynb`、0始まりのindex=11。コード内容は変更していません。

In [ ]:
# ============================================================
# USD/JPY 5分足
#
# 時間減衰ウェイト
# + 二段階AI
# + 動的閾値
# + ボラティリティフィルター
# + Walk-Forward
#
# ============================================================


# ============================================================
# 1. ライブラリ
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 2. 基本設定
# ============================================================

SYMBOL = "JPY=X"

PERIOD = "60d"

INTERVAL = "5m"


# 30分後を見る
HOLD_BARS = 6


# ±0.05%以上なら「MOVE」
MOVE_THRESHOLD = 0.0005


# 仮の取引コスト
TRADING_COST = 0.0000133


# 学習とテストの間のGap
GAP = HOLD_BARS


# 古いデータの重みを減らす速度
#
# half_life = 20日
#
# 20日前のデータは
# 最新データの約半分の重みになる
HALF_LIFE_DAYS = 20


# ============================================================
# 3. データ取得
# ============================================================

df = yf.download(
    SYMBOL,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    progress=False
)


# yfinanceの列が2段なら1段にする
if df.columns.nlevels > 1:

    df.columns = (
        df.columns
        .get_level_values(0)
    )


# 日本時間へ
if df.index.tz is not None:

    df.index = (
        df.index
        .tz_convert(
            "Asia/Tokyo"
        )
    )


print(
    "取得した5分足:",
    len(df)
)


# ============================================================
# 4. リターン
# ============================================================

df["return_5m"] = (
    df["Close"]
    .pct_change(1)
)

df["return_15m"] = (
    df["Close"]
    .pct_change(3)
)

df["return_30m"] = (
    df["Close"]
    .pct_change(6)
)

df["return_1h"] = (
    df["Close"]
    .pct_change(12)
)

df["return_2h"] = (
    df["Close"]
    .pct_change(24)
)


# ============================================================
# 5. 移動平均
# ============================================================

df["MA5"] = (
    df["Close"]
    .rolling(5)
    .mean()
)

df["MA20"] = (
    df["Close"]
    .rolling(20)
    .mean()
)

df["MA50"] = (
    df["Close"]
    .rolling(50)
    .mean()
)


# 現在価格とMAの距離
df["MA5_distance"] = (
    df["Close"]
    / df["MA5"]
    - 1
)

df["MA20_distance"] = (
    df["Close"]
    / df["MA20"]
    - 1
)

df["MA50_distance"] = (
    df["Close"]
    / df["MA50"]
    - 1
)


# MAの傾き
df["MA5_slope"] = (
    df["MA5"]
    .pct_change(3)
)

df["MA20_slope"] = (
    df["MA20"]
    .pct_change(3)
)

df["MA50_slope"] = (
    df["MA50"]
    .pct_change(3)
)


# ============================================================
# 6. ローソク足
# ============================================================

df["body"] = (

    abs(
        df["Close"]
        - df["Open"]
    )

    / df["Open"]
)


df["range"] = (

    df["High"]
    - df["Low"]

) / df["Close"]


df["upper_wick"] = (

    df["High"]

    - df[
        [
            "Open",
            "Close"
        ]
    ].max(
        axis=1
    )

) / df["Close"]


df["lower_wick"] = (

    df[
        [
            "Open",
            "Close"
        ]
    ].min(
        axis=1
    )

    - df["Low"]

) / df["Close"]


df["bullish"] = (

    df["Close"]
    > df["Open"]

).astype(int)


# ============================================================
# 7. ボラティリティ
# ============================================================

df["volatility_1h"] = (

    df["return_5m"]
    .rolling(12)
    .std()
)

df["volatility_2h"] = (

    df["return_5m"]
    .rolling(24)
    .std()
)

df["volatility_4h"] = (

    df["return_5m"]
    .rolling(48)
    .std()
)


# ============================================================
# 8. RSI
# ============================================================

delta = (
    df["Close"]
    .diff()
)


gain = (
    delta
    .clip(
        lower=0
    )
)


loss = (
    -delta
    .clip(
        upper=0
    )
)


avg_gain = (
    gain
    .rolling(14)
    .mean()
)


avg_loss = (
    loss
    .rolling(14)
    .mean()
)


rs = (
    avg_gain
    / avg_loss
)


df["RSI"] = (

    100
    - 100
    / (
        1 + rs
    )
)


# ============================================================
# 9. 高値・安値
# ============================================================

df["high_1h"] = (

    df["High"]
    .rolling(12)
    .max()
)


df["low_1h"] = (

    df["Low"]
    .rolling(12)
    .min()
)


df["distance_high_1h"] = (

    df["Close"]
    / df["high_1h"]
    - 1
)


df["distance_low_1h"] = (

    df["Close"]
    / df["low_1h"]
    - 1
)


# ============================================================
# 10. 時間特徴
# ============================================================

df["hour"] = (
    df.index.hour
)

df["weekday"] = (
    df.index.dayofweek
)


df["hour_sin"] = np.sin(

    2
    * np.pi
    * df["hour"]
    / 24
)


df["hour_cos"] = np.cos(

    2
    * np.pi
    * df["hour"]
    / 24
)


# ============================================================
# 11. 実際の売買に近い将来リターン
# ============================================================

# 現在足が確定
#
# ↓
#
# 次の足Openで買う

df["entry_price"] = (
    df["Open"]
    .shift(-1)
)


# 30分後
df["exit_price"] = (
    df["Close"]
    .shift(-HOLD_BARS)
)


df["future_return"] = (

    df["exit_price"]
    / df["entry_price"]
    - 1
)


# ============================================================
# 12. MOVEターゲット
# ============================================================

df["move_target"] = (

    abs(
        df["future_return"]
    )

    > MOVE_THRESHOLD

).astype(int)


# ============================================================
# 13. Directionターゲット
# ============================================================

df["direction_target"] = np.where(

    df["future_return"] > 0,

    1,

    0
)


# ============================================================
# 14. MOVEモデルの特徴量
# ============================================================

move_features = [

    "volatility_1h",
    "volatility_2h",
    "volatility_4h",

    "range",
    "body",

    "return_5m",
    "return_15m",
    "return_30m",

    "MA20_slope",
    "MA50_slope",

    "distance_high_1h",
    "distance_low_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


# ============================================================
# 15. Directionモデルの特徴量
# ============================================================

direction_features = [

    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",

    "MA5_distance",
    "MA20_distance",
    "MA50_distance",

    "MA5_slope",
    "MA20_slope",
    "MA50_slope",

    "RSI",

    "bullish",

    "body",
    "upper_wick",
    "lower_wick",

    "distance_high_1h",
    "distance_low_1h",

    "volatility_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


# ============================================================
# 16. データ作成
# ============================================================

all_features = list(
    set(
        move_features
        + direction_features
    )
)


required_columns = (

    all_features

    + [

        "move_target",
        "direction_target",

        "future_return",

        "entry_price",
        "exit_price"

    ]
)


data = (

    df[
        required_columns
    ]

    .dropna()

    .copy()
)


print(
    "学習可能データ:",
    len(data)
)


# ============================================================
# 17. 時間減衰ウェイトを作る関数
# ============================================================

def make_time_weights(
    index,
    half_life_days
):

    # 最新日時
    latest_time = (
        index.max()
    )


    # 最新から何日前か
    age_days = (

        latest_time
        - index

    ).total_seconds() / 86400


    # 指数減衰
    weights = (

        0.5
        ** (
            age_days
            / half_life_days
        )

    )


    return np.array(
        weights
    )


# ============================================================
# 18. 動的閾値を探す関数
# ============================================================

def find_best_thresholds(
    validation_data,
    p_move,
    p_up,
    p_down
):


    # 候補
    move_thresholds = [

        0.55,
        0.60,
        0.65,
        0.70
    ]


    direction_thresholds = [

        0.55,
        0.60,
        0.65,
        0.70
    ]


    best_score = -999

    best_move = 0.65
    best_direction = 0.60


    for move_t in move_thresholds:

        for direction_t in direction_thresholds:


            # BUY
            buy_mask = (

                (p_move >= move_t)

                &

                (p_up >= direction_t)

                &

                (
                    p_up
                    > p_down
                )
            )


            # SELL
            sell_mask = (

                (p_move >= move_t)

                &

                (p_down >= direction_t)

                &

                (
                    p_down
                    > p_up
                )
            )


            returns = []


            # BUYリターン
            if buy_mask.sum() > 0:

                buy_returns = (

                    validation_data[
                        "future_return"
                    ].values[
                        buy_mask
                    ]

                    - TRADING_COST
                )

                returns.extend(
                    buy_returns
                )


            # SELL
            if sell_mask.sum() > 0:

                sell_returns = (

                    -validation_data[
                        "future_return"
                    ].values[
                        sell_mask
                    ]

                    - TRADING_COST
                )

                returns.extend(
                    sell_returns
                )


            returns = np.array(
                returns
            )


            # 取引数が少なすぎる設定は除外
            if len(returns) < 20:

                continue


            # 平均リターン
            score = (
                returns.mean()
            )


            if score > best_score:

                best_score = score

                best_move = move_t

                best_direction = direction_t


    return (
        best_move,
        best_direction,
        best_score
    )


# ============================================================
# 19. Walk-Forward設定
# ============================================================

N_SPLITS = 5


block_size = (

    len(data)

    // (
        N_SPLITS + 1
    )
)


walk_results = []

all_trades = []


# ============================================================
# 20. Walk-Forward開始
# ============================================================

for fold in range(
    N_SPLITS
):


    # ----------------------------------------
    # 学習 / テスト範囲
    # ----------------------------------------

    train_end = (

        block_size
        * (
            fold + 1
        )
    )


    test_start = (

        train_end
        + GAP
    )


    test_end = (

        test_start
        + block_size
    )


    if test_end > len(data):

        test_end = len(data)


    train_full = (

        data.iloc[
            :train_end
        ]
    )


    test_fold = (

        data.iloc[
            test_start:test_end
        ]
    )


    if len(test_fold) == 0:

        continue


    # ========================================================
    # 21. 学習期間の最後20%をValidationにする
    # ========================================================

    validation_split = int(

        len(train_full)
        * 0.80

    )


    train_inner = (

        train_full.iloc[
            :validation_split
        ]
    )


    validation = (

        train_full.iloc[
            validation_split:
        ]
    )


    # ========================================================
    # 22. 時間減衰ウェイト
    # ========================================================

    move_weights = (
        make_time_weights(

            train_inner.index,

            HALF_LIFE_DAYS
        )
    )


    # ========================================================
    # 23. MOVEモデル
    # ========================================================

    move_model = RandomForestClassifier(

        n_estimators=400,

        max_depth=8,

        min_samples_leaf=20,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    move_model.fit(

        train_inner[
            move_features
        ],

        train_inner[
            "move_target"
        ],

        sample_weight=
            move_weights
    )


    # ========================================================
    # 24. Directionモデル学習
    # ========================================================

    direction_train_inner = (

        train_inner[

            train_inner[
                "move_target"
            ] == 1

        ]
    )


    direction_weights = (
        make_time_weights(

            direction_train_inner.index,

            HALF_LIFE_DAYS
        )
    )


    direction_model = RandomForestClassifier(

        n_estimators=400,

        max_depth=8,

        min_samples_leaf=15,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    direction_model.fit(

        direction_train_inner[
            direction_features
        ],

        direction_train_inner[
            "direction_target"
        ],

        sample_weight=
            direction_weights
    )


    # ========================================================
    # 25. Validationで確率を出す
    # ========================================================

    validation_move_prob = (

        move_model
        .predict_proba(

            validation[
                move_features
            ]

        )[:, 1]
    )


    validation_direction_prob = (

        direction_model
        .predict_proba(

            validation[
                direction_features
            ]

        )
    )


    class_map = {

        c: i

        for i, c

        in enumerate(

            direction_model.classes_

        )
    }


    validation_p_down = (

        validation_direction_prob[
            :,
            class_map[0]
        ]
    )


    validation_p_up = (

        validation_direction_prob[
            :,
            class_map[1]
        ]
    )


    # ========================================================
    # 26. Validationだけを使って閾値決定
    # ========================================================

    best_move_threshold, \
    best_direction_threshold, \
    validation_score = (

        find_best_thresholds(

            validation,

            validation_move_prob,

            validation_p_up,

            validation_p_down
        )

    )


    # ========================================================
    # 27. ボラティリティフィルタも学習期間だけで決定
    # ========================================================

    vol_low = (

        train_full[
            "volatility_1h"
        ]
        .quantile(
            0.20
        )
    )


    vol_high = (

        train_full[
            "volatility_1h"
        ]
        .quantile(
            0.80
        )
    )


    # ========================================================
    # 28. 全学習データで再学習
    # ========================================================

    move_weights_full = (
        make_time_weights(

            train_full.index,

            HALF_LIFE_DAYS
        )
    )


    final_move_model = RandomForestClassifier(

        n_estimators=500,

        max_depth=8,

        min_samples_leaf=20,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    final_move_model.fit(

        train_full[
            move_features
        ],

        train_full[
            "move_target"
        ],

        sample_weight=
            move_weights_full
    )


    direction_train_full = (

        train_full[

            train_full[
                "move_target"
            ] == 1

        ]
    )


    direction_weights_full = (
        make_time_weights(

            direction_train_full.index,

            HALF_LIFE_DAYS
        )
    )


    final_direction_model = RandomForestClassifier(

        n_estimators=500,

        max_depth=8,

        min_samples_leaf=15,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    final_direction_model.fit(

        direction_train_full[
            direction_features
        ],

        direction_train_full[
            "direction_target"
        ],

        sample_weight=
            direction_weights_full
    )


    # ========================================================
    # 29. 完全未知テストに予測
    # ========================================================

    test_move_prob = (

        final_move_model
        .predict_proba(

            test_fold[
                move_features
            ]

        )[:, 1]
    )


    test_direction_prob = (

        final_direction_model
        .predict_proba(

            test_fold[
                direction_features
            ]

        )
    )


    class_map_test = {

        c: i

        for i, c

        in enumerate(

            final_direction_model.classes_

        )
    }


    test_p_down = (

        test_direction_prob[
            :,
            class_map_test[0]
        ]
    )


    test_p_up = (

        test_direction_prob[
            :,
            class_map_test[1]
        ]
    )


    # ========================================================
    # 30. ボラティリティフィルター
    # ========================================================

    vol_filter = (

        (
            test_fold[
                "volatility_1h"
            ].values
            >= vol_low
        )

        &

        (
            test_fold[
                "volatility_1h"
            ].values
            <= vol_high
        )
    )


    # ========================================================
    # 31. BUY / SELLシグナル
    # ========================================================

    buy_signal = (

        vol_filter

        &

        (
            test_move_prob
            >= best_move_threshold
        )

        &

        (
            test_p_up
            >= best_direction_threshold
        )

        &

        (
            test_p_up
            > test_p_down
        )
    )


    sell_signal = (

        vol_filter

        &

        (
            test_move_prob
            >= best_move_threshold
        )

        &

        (
            test_p_down
            >= best_direction_threshold
        )

        &

        (
            test_p_down
            > test_p_up
        )
    )


    signals = np.zeros(
        len(test_fold)
    )


    signals[
        buy_signal
    ] = 1


    signals[
        sell_signal
    ] = -1


    # ========================================================
    # 32. 非重複バックテスト
    # ========================================================

    returns = []


    i = 0


    while i < len(
        test_fold
    ):


        signal = (
            signals[i]
        )


        if signal == 0:

            i += 1

            continue


        row = (
            test_fold.iloc[i]
        )


        # BUY
        if signal == 1:

            trade_return = (

                row[
                    "future_return"
                ]

                - TRADING_COST
            )

            direction = "BUY"


        # SELL
        else:

            trade_return = (

                -row[
                    "future_return"
                ]

                - TRADING_COST
            )

            direction = "SELL"


        returns.append(
            trade_return
        )


        all_trades.append({

            "fold":
                fold + 1,

            "time":
                test_fold.index[i],

            "direction":
                direction,

            "return":
                trade_return,

            "p_move":
                test_move_prob[i],

            "p_up":
                test_p_up[i],

            "p_down":
                test_p_down[i]
        })


        # 30分保有
        i += HOLD_BARS


    returns = np.array(
        returns
    )


    # ========================================================
    # 33. Fold結果
    # ========================================================

    if len(returns) > 0:

        win_rate = (

            returns
            > 0

        ).mean()


        average_return = (
            returns.mean()
        )


    else:

        win_rate = np.nan

        average_return = np.nan


    walk_results.append({

        "fold":
            fold + 1,

        "train_size":
            len(train_full),

        "test_size":
            len(test_fold),

        "move_threshold":
            best_move_threshold,

        "direction_threshold":
            best_direction_threshold,

        "vol_low":
            vol_low,

        "vol_high":
            vol_high,

        "trades":
            len(returns),

        "win_rate":
            win_rate,

        "average_return":
            average_return
    })


# ============================================================
# 34. Walk-Forward結果
# ============================================================

walk_df = pd.DataFrame(
    walk_results
)


walk_df[
    "win_rate"
] *= 100


walk_df[
    "average_return"
] *= 100


print(
    "\n=============================="
)

print(
    "時間減衰 Walk-Forward"
)

print(
    "=============================="
)


print(
    walk_df
)


# ============================================================
# 35. 平均成績
# ============================================================

print(
    "\n=============================="
)

print(
    "Walk-Forward総合"
)

print(
    "=============================="
)


print(

    "平均勝率:",

    round(
        walk_df[
            "win_rate"
        ].mean(),
        2
    ),

    "%"
)


print(

    "平均リターン:",

    round(
        walk_df[
            "average_return"
        ].mean(),
        4
    ),

    "%"
)


print(

    "合計取引数:",

    walk_df[
        "trades"
    ].sum()
)


# ============================================================
# 36. 全取引
# ============================================================

trades_df = pd.DataFrame(
    all_trades
)


if len(
    trades_df
) > 0:


    # 累積損益
    trades_df[
        "equity"
    ] = (

        1

        + trades_df[
            "return"
        ]

    ).cumprod()


    plt.figure(
        figsize=(
            14,
            6
        )
    )


    plt.plot(

        trades_df[
            "time"
        ],

        trades_df[
            "equity"
        ]
    )


    plt.title(
        "Time Weighted Walk Forward Equity"
    )


    plt.xlabel(
        "Time"
    )


    plt.ylabel(
        "Growth of 1"
    )


    plt.grid()


    plt.show()


# ============================================================
# 37. BUY / SELL別
# ============================================================

if len(
    trades_df
) > 0:


    print(
        "\n=============================="
    )

    print(
        "BUY / SELL"
    )

    print(
        "=============================="
    )


    for direction in [

        "BUY",
        "SELL"

    ]:


        subset = (

            trades_df[

                trades_df[
                    "direction"
                ]
                == direction

            ]

        )


        if len(
            subset
        ) == 0:

            continue


        print(
            "\n",
            direction
        )


        print(

            "件数:",

            len(
                subset
            )
        )


        print(

            "勝率:",

            round(

                (
                    subset[
                        "return"
                    ] > 0
                ).mean()
                * 100,

                2

            ),

            "%"
        )


        print(

            "平均リターン:",

            round(

                subset[
                    "return"
                ].mean()
                * 100,

                4

            ),

            "%"
        )

## 元Notebookのセル 13

出典: `FX.ipynb`、0始まりのindex=12。コード内容は変更していません。

In [ ]:
# ============================================================
# MFE / MAE + 保有時間 + TP/SL + 資金シミュレーション
# 前提:
# 前のコードで trades_df, data, df が作成済み
# ============================================================


# ============================================================
# 1. 基本設定
# ============================================================

INITIAL_CAPITAL = 10000

LEVERAGES = [
    1,
    3,
    5
]

# 比較する保有時間
HOLD_MINUTES_LIST = [
    5,
    10,
    15,
    20,
    30,
    45,
    60
]

# 5分足なので分→本数
HOLD_BARS_LIST = {
    minutes: minutes // 5
    for minutes in HOLD_MINUTES_LIST
}

# TP候補
TP_LIST = [
    0.0003,   # 0.03%
    0.0005,   # 0.05%
    0.0008,   # 0.08%
    0.0010    # 0.10%
]

# SL候補
SL_LIST = [
    0.0003,   # 0.03%
    0.0005,   # 0.05%
    0.0007,   # 0.07%
    0.0010    # 0.10%
]

# 仮の取引コスト
TRADING_COST = 0.0000133


# ============================================================
# 2. MFE / MAEを計算する関数
# ============================================================

def calculate_mfe_mae(
    signal_time,
    direction,
    entry_price,
    hold_bars=6
):

    # シグナル時刻の位置
    try:
        start_loc = df.index.get_loc(
            signal_time
        )
    except KeyError:
        return np.nan, np.nan


    # 実際のエントリーは次の足
    entry_loc = start_loc + 1

    end_loc = (
        entry_loc
        + hold_bars
    )


    # データ不足なら終了
    if end_loc >= len(df):
        return np.nan, np.nan


    future_window = (
        df.iloc[
            entry_loc:end_loc + 1
        ]
    )


    # BUYの場合
    if direction == "BUY":

        max_favorable = (
            future_window["High"].max()
            / entry_price
            - 1
        )

        max_adverse = (
            future_window["Low"].min()
            / entry_price
            - 1
        )


    # SELLの場合
    else:

        max_favorable = (
            entry_price
            / future_window["Low"].min()
            - 1
        )

        max_adverse = (
            entry_price
            / future_window["High"].max()
            - 1
        )


    return (
        max_favorable,
        max_adverse
    )


# ============================================================
# 3. 既存取引にMFE / MAEを追加
# ============================================================

mfe_values = []
mae_values = []


for _, trade in trades_df.iterrows():

    mfe, mae = calculate_mfe_mae(

        signal_time=
            trade["time"],

        direction=
            trade["direction"],

        entry_price=
            df.loc[
                trade["time"],
                "Open"
            ]
            if trade["time"] in df.index
            else np.nan,

        hold_bars=6
    )

    mfe_values.append(
        mfe
    )

    mae_values.append(
        mae
    )


trades_df[
    "MFE_30m"
] = mfe_values

trades_df[
    "MAE_30m"
] = mae_values


# ============================================================
# 4. MFE / MAE概要
# ============================================================

print(
    "\n=============================="
)

print(
    "MFE / MAE 概要"
)

print(
    "=============================="
)


print(
    "平均MFE:",
    round(
        trades_df[
            "MFE_30m"
        ].mean()
        * 100,
        4
    ),
    "%"
)


print(
    "中央値MFE:",
    round(
        trades_df[
            "MFE_30m"
        ].median()
        * 100,
        4
    ),
    "%"
)


print(
    "平均MAE:",
    round(
        trades_df[
            "MAE_30m"
        ].mean()
        * 100,
        4
    ),
    "%"
)


print(
    "中央値MAE:",
    round(
        trades_df[
            "MAE_30m"
        ].median()
        * 100,
        4
    ),
    "%"
)


# ============================================================
# 5. 勝ち / 負け別MFE・MAE
# ============================================================

for name, subset in [

    (
        "WIN",
        trades_df[
            trades_df[
                "return"
            ] > 0
        ]
    ),

    (
        "LOSS",
        trades_df[
            trades_df[
                "return"
            ] <= 0
        ]
    )

]:

    if len(subset) == 0:
        continue


    print(
        "\n",
        name
    )

    print(
        "件数:",
        len(subset)
    )

    print(
        "平均MFE:",
        round(
            subset[
                "MFE_30m"
            ].mean()
            * 100,
            4
        ),
        "%"
    )

    print(
        "平均MAE:",
        round(
            subset[
                "MAE_30m"
            ].mean()
            * 100,
            4
        ),
        "%"
    )


# ============================================================
# 6. BUY / SELL別 MFE・MAE
# ============================================================

for direction in [
    "BUY",
    "SELL"
]:

    subset = (
        trades_df[
            trades_df[
                "direction"
            ] == direction
        ]
    )

    if len(subset) == 0:
        continue


    print(
        "\n=============================="
    )

    print(
        direction,
        "MFE / MAE"
    )

    print(
        "=============================="
    )

    print(
        "件数:",
        len(subset)
    )

    print(
        "平均MFE:",
        round(
            subset[
                "MFE_30m"
            ].mean()
            * 100,
            4
        ),
        "%"
    )

    print(
        "平均MAE:",
        round(
            subset[
                "MAE_30m"
            ].mean()
            * 100,
            4
        ),
        "%"
    )


# ============================================================
# 7. 保有時間別リターンを計算する関数
# ============================================================

def calculate_return_by_holding(
    signal_time,
    direction,
    hold_bars
):

    try:
        signal_loc = (
            df.index
            .get_loc(
                signal_time
            )
        )

    except KeyError:
        return np.nan


    entry_loc = (
        signal_loc + 1
    )

    exit_loc = (
        entry_loc
        + hold_bars
    )


    if exit_loc >= len(df):
        return np.nan


    entry_price = (
        df.iloc[
            entry_loc
        ]["Open"]
    )

    exit_price = (
        df.iloc[
            exit_loc
        ]["Close"]
    )


    if direction == "BUY":

        r = (
            exit_price
            / entry_price
            - 1
        )

    else:

        r = (
            entry_price
            / exit_price
            - 1
        )


    return (
        r
        - TRADING_COST
    )


# ============================================================
# 8. 保有時間比較
# ============================================================

holding_results = []


for minutes, bars in HOLD_BARS_LIST.items():

    returns = []


    for _, trade in trades_df.iterrows():

        r = calculate_return_by_holding(

            signal_time=
                trade["time"],

            direction=
                trade["direction"],

            hold_bars=
                bars
        )


        if not np.isnan(r):

            returns.append(
                r
            )


    returns = np.array(
        returns
    )


    if len(returns) == 0:
        continue


    holding_results.append({

        "minutes":
            minutes,

        "trades":
            len(returns),

        "win_rate":
            (
                returns > 0
            ).mean(),

        "average_return":
            returns.mean(),

        "median_return":
            np.median(
                returns
            )
    })


holding_df = pd.DataFrame(
    holding_results
)


holding_df[
    "win_rate"
] *= 100

holding_df[
    "average_return"
] *= 100

holding_df[
    "median_return"
] *= 100


print(
    "\n=============================="
)

print(
    "保有時間比較"
)

print(
    "=============================="
)

print(
    holding_df
)


# ============================================================
# 9. TP / SLバックテスト関数
# ============================================================

def simulate_tp_sl(
    signal_time,
    direction,
    tp,
    sl,
    max_hold_bars=6
):

    try:

        signal_loc = (
            df.index
            .get_loc(
                signal_time
            )
        )

    except KeyError:

        return np.nan


    entry_loc = (
        signal_loc + 1
    )


    if entry_loc >= len(df):

        return np.nan


    entry_price = (
        df.iloc[
            entry_loc
        ]["Open"]
    )


    # BUY
    if direction == "BUY":

        tp_price = (
            entry_price
            * (
                1 + tp
            )
        )

        sl_price = (
            entry_price
            * (
                1 - sl
            )
        )


    # SELL
    else:

        tp_price = (
            entry_price
            * (
                1 - tp
            )
        )

        sl_price = (
            entry_price
            * (
                1 + sl
            )
        )


    # ----------------------------------------
    # 各5分足を順番に確認
    # ----------------------------------------

    for j in range(
        1,
        max_hold_bars + 1
    ):

        loc = (
            entry_loc
            + j
        )


        if loc >= len(df):
            break


        high = (
            df.iloc[
                loc
            ]["High"]
        )

        low = (
            df.iloc[
                loc
            ]["Low"]
        )


        # BUY
        if direction == "BUY":

            tp_hit = (
                high
                >= tp_price
            )

            sl_hit = (
                low
                <= sl_price
            )


        # SELL
        else:

            tp_hit = (
                low
                <= tp_price
            )

            sl_hit = (
                high
                >= sl_price
            )


        # ----------------------------------------
        # 同じ5分足でTP/SL両方に触れた場合
        # 順番が分からないので保守的にSL扱い
        # ----------------------------------------

        if tp_hit and sl_hit:

            return (
                -sl
                - TRADING_COST
            )


        if tp_hit:

            return (
                tp
                - TRADING_COST
            )


        if sl_hit:

            return (
                -sl
                - TRADING_COST
            )


    # ----------------------------------------
    # どちらにも触れなければ
    # max_hold_bars後のCloseで決済
    # ----------------------------------------

    exit_loc = (
        entry_loc
        + max_hold_bars
    )


    if exit_loc >= len(df):

        return np.nan


    exit_price = (
        df.iloc[
            exit_loc
        ]["Close"]
    )


    if direction == "BUY":

        r = (
            exit_price
            / entry_price
            - 1
        )

    else:

        r = (
            entry_price
            / exit_price
            - 1
        )


    return (
        r
        - TRADING_COST
    )


# ============================================================
# 10. TP / SL 全組み合わせ
# ============================================================

tp_sl_results = []


for tp in TP_LIST:

    for sl in SL_LIST:

        returns = []


        for _, trade in trades_df.iterrows():

            r = simulate_tp_sl(

                signal_time=
                    trade["time"],

                direction=
                    trade["direction"],

                tp=
                    tp,

                sl=
                    sl,

                max_hold_bars=6
            )


            if not np.isnan(r):

                returns.append(
                    r
                )


        returns = np.array(
            returns
        )


        if len(returns) == 0:
            continue


        wins = (
            returns[
                returns > 0
            ]
        )

        losses = (
            returns[
                returns < 0
            ]
        )


        total_profit = (
            wins.sum()
            if len(wins) > 0
            else 0
        )

        total_loss = (
            abs(
                losses.sum()
            )
            if len(losses) > 0
            else 0
        )


        if total_loss > 0:

            profit_factor = (
                total_profit
                / total_loss
            )

        else:

            profit_factor = np.nan


        tp_sl_results.append({

            "tp":
                tp,

            "sl":
                sl,

            "trades":
                len(returns),

            "win_rate":
                (
                    returns > 0
                ).mean(),

            "average_return":
                returns.mean(),

            "median_return":
                np.median(
                    returns
                ),

            "profit_factor":
                profit_factor
        })


tp_sl_df = pd.DataFrame(
    tp_sl_results
)


tp_sl_df[
    "tp_pct"
] = (
    tp_sl_df["tp"]
    * 100
)

tp_sl_df[
    "sl_pct"
] = (
    tp_sl_df["sl"]
    * 100
)

tp_sl_df[
    "win_rate"
] *= 100

tp_sl_df[
    "average_return"
] *= 100

tp_sl_df[
    "median_return"
] *= 100


# 平均リターン順
tp_sl_df = (
    tp_sl_df
    .sort_values(
        "average_return",
        ascending=False
    )
)


print(
    "\n=============================="
)

print(
    "TP / SL 比較"
)

print(
    "=============================="
)

print(
    tp_sl_df[
        [
            "tp_pct",
            "sl_pct",
            "trades",
            "win_rate",
            "average_return",
            "median_return",
            "profit_factor"
        ]
    ]
)


# ============================================================
# 11. 基本損益統計
# ============================================================

base_returns = (
    trades_df[
        "return"
    ].dropna()
)


wins = (
    base_returns[
        base_returns > 0
    ]
)

losses = (
    base_returns[
        base_returns < 0
    ]
)


overall_win_rate = (
    (
        base_returns > 0
    ).mean()
)


average_win = (
    wins.mean()
    if len(wins) > 0
    else np.nan
)


average_loss = (
    losses.mean()
    if len(losses) > 0
    else np.nan
)


payoff_ratio = (
    average_win
    / abs(
        average_loss
    )
    if (
        not np.isnan(
            average_win
        )
        and
        not np.isnan(
            average_loss
        )
        and
        average_loss != 0
    )
    else np.nan
)


expected_value = (
    base_returns.mean()
)


total_profit = (
    wins.sum()
    if len(wins) > 0
    else 0
)


total_loss = (
    abs(
        losses.sum()
    )
    if len(losses) > 0
    else 0
)


profit_factor = (
    total_profit
    / total_loss
    if total_loss > 0
    else np.nan
)


print(
    "\n=============================="
)

print(
    "詳細損益統計"
)

print(
    "=============================="
)


print(
    "総取引数:",
    len(
        base_returns
    )
)


print(
    "全取引勝率:",
    round(
        overall_win_rate
        * 100,
        2
    ),
    "%"
)


print(
    "平均利益:",
    round(
        average_win
        * 100,
        4
    ),
    "%"
)


print(
    "平均損失:",
    round(
        average_loss
        * 100,
        4
    ),
    "%"
)


print(
    "Payoff Ratio:",
    round(
        payoff_ratio,
        3
    )
)


print(
    "期待値 / 取引:",
    round(
        expected_value
        * 100,
        4
    ),
    "%"
)


print(
    "Profit Factor:",
    round(
        profit_factor,
        3
    )
)


# ============================================================
# 12. 最大ドローダウン
# ============================================================

equity = (
    1
    + base_returns
).cumprod()


running_max = (
    equity
    .cummax()
)


drawdown = (
    equity
    / running_max
    - 1
)


max_drawdown = (
    drawdown.min()
)


print(
    "最大DD:",
    round(
        max_drawdown
        * 100,
        4
    ),
    "%"
)


# ============================================================
# 13. 1万円・レバレッジ別シミュレーション
# ============================================================

capital_results = []


for leverage in LEVERAGES:

    capital = (
        INITIAL_CAPITAL
    )


    capital_curve = [
        capital
    ]


    for r in base_returns:

        # レバレッジを掛ける
        leveraged_return = (
            r
            * leverage
        )


        capital *= (
            1
            + leveraged_return
        )


        capital_curve.append(
            capital
        )


    total_profit_yen = (
        capital
        - INITIAL_CAPITAL
    )


    capital_results.append({

        "leverage":
            leverage,

        "start_capital":
            INITIAL_CAPITAL,

        "end_capital":
            capital,

        "profit_yen":
            total_profit_yen,

        "return_pct":
            (
                capital
                / INITIAL_CAPITAL
                - 1
            )
            * 100
    })


capital_df = pd.DataFrame(
    capital_results
)


print(
    "\n=============================="
)

print(
    "1万円 資金シミュレーション"
)

print(
    "=============================="
)

print(
    capital_df
)


# ============================================================
# 14. 1取引あたり期待利益（円）
# ============================================================

print(
    "\n=============================="
)

print(
    "1取引あたり期待利益"
)

print(
    "=============================="
)


for leverage in LEVERAGES:

    expected_yen = (

        INITIAL_CAPITAL

        * leverage

        * expected_value
    )


    print(

        f"{leverage}倍:",

        round(
            expected_yen,
            2
        ),

        "円 / 取引"
    )


# ============================================================
# 15. 仮に1日5回 / 10回なら
# ============================================================

print(
    "\n=============================="
)

print(
    "1日期待利益の概算"
)

print(
    "=============================="
)


for leverage in LEVERAGES:

    expected_yen = (

        INITIAL_CAPITAL

        * leverage

        * expected_value
    )


    print(
        f"\nレバレッジ {leverage}倍"
    )


    print(
        "1日5取引:",
        round(
            expected_yen
            * 5,
            2
        ),
        "円"
    )


    print(
        "1日10取引:",
        round(
            expected_yen
            * 10,
            2
        ),
        "円"
    )